# Signpost-D compute probe: does the model fail to compute, or fail to map a value to an option?

Score questions that need a computed value (a ratio, a time difference, a percentage of a count) are where the decoder loses most. This notebook separates the two explanations, with no training, on questions never trained on:

- **Arm A (as now):** one forward pass per case, the letter read after `n:` exactly as notebook 11 does.
- **Arm B (value given):** as A, with the computed value added to the case as one line ("departure is 3 days after arrival"). Only on the code-built questions, where the value is known by code (dates, ratios, differences).
- **Arm C (reason, then answer; only on `C_SETS`):** the model writes its working, then an `Answers:` block (`1: B`); the letters are parsed from the text. Greedy decoding.

Every case is timed at batch 1 (one request, as in use): one forward pass for A and B, the whole generation for C.

**Sets** (each case keeps only the questions of its set, the same in every arm):
- `numq_computed`: held-out code-built true/false questions that need a date gap, a ratio or a difference (96); arms A, B, C.
- `numq_band`: held-out code-built score questions whose value is stated and only has to be placed in a band (101); A, C.
- `dev_score_compute`: our dev's score questions of kind compare or infer (291); A on all, C on `C_MAX_CASES` cases.
- `dev_score_judge`: our dev's score questions of kind judge or read, the control; A on all, C on `C_MAX_CASES` cases.

**Models:** the untrained base (`base`) and notebook 11's A5 adapter merged in (`tuned`). The tuned model's adapter is read from notebook 11's `checkpoints/` in the working area; if it is not there, the tuned runs are skipped.

**Item 1a (added 28 Oct): `shift_full`** = every shift_dev case with all its questions in one pass, arm A only (base and tuned). It reports our misses by how sure the reviewers' gold is (clear: top ≥ 0.85; leaning: 0.6–0.85; split: under 0.6), and the time per question on full cases against the 100 ms budget. If most misses sit on clear gold, relabelling with a teacher model cannot help; if many sit on leaning or split gold, it might.

**Reading the result:** B much better than A means computing in one pass is the bottleneck. C better than A on `dev_score_compute` but not on `dev_score_judge` points the same way, and C's latency is the price. B no better than A means the problem is mapping a value to the levels. No benchmark scoring; nothing is uploaded.

In [1]:
# Parameters

MODEL = {"name": "Qwen/Qwen3-4B-Instruct-2507", "weights": "fp16"}
TUNED_ADAPTER = "dec-Qwen3-4B-Instruct-2507-v1_6-train-pos1/lr0.0001-r16-ep2-fp16-packed-seed42/adapter"   # A5, under checkpoints/ in the working area
RUNS = [("base", "A"), ("base", "B"), ("base", "C"), ("tuned", "A"), ("tuned", "B")]   # (model, arm); base runs first
SETS = ["numq_computed", "numq_band", "dev_score_compute", "dev_score_judge", "shift_full"]   # shift_full: item 1a
C_SETS = ["numq_computed"]           # arm C only on these sets (about 40 s a case on a T4: a diagnostic, not a usable latency)
C_MAX_CASES = 60                      # arm C on at most this many cases per set (a fixed sample; A is also reported on it)
MAX_NEW_TOKENS = 1536                 # arm C: working plus answers, per case
GPUS = 2                              # cases shared out over the GPUs; each case is timed on its own GPU at batch 1
D_SET = "v1_6"                        # our dev split as in notebook 11
BUDGET_MS = 100                       # per question, T4, batch 1
RUN_NAME = "12_compute_probe"

In [2]:
# Cell 1 — Setup: packages, fixed settings, progress log

!pip install -q --force-reinstall --no-deps "transformers==5.17.0"

import datetime, gc, json, math, os, random, re, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"   # GPU checks through NVML: CUDA must not start before the GPU processes are forked
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import transformers
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get("HF_TOKEN") or None     # never printed or written to the notebook
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        for label in ("HF_TOKEN", "hf_token"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(label) or None
                break
            except Exception:
                pass
    except ImportError:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
MAX_PROMPT_TOKENS = 4096
LETTERS = [chr(65 + i) for i in range(26)]
DEV_FRACTION = 0.15
OUT = Path(RUN_NAME)
OUT.mkdir(exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
RANK, WORLD = 0, 1
AMP = DEVICE.type == "cuda"
RUN_LOG = Path(f"{RUN_NAME}_run.log")
assert transformers.__version__ == "5.17.0", "restart the kernel and run from the top"
api = HfApi()


def log(message):
    if RANK:
        return
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


log(f"{RUN_NAME} | runs {RUNS} | sets {SETS} | C on up to {C_MAX_CASES} cases per set | torch {torch.__version__} | "
    f"GPUs visible {torch.cuda.device_count()} (using up to {GPUS})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 66.7 MB/s eta 0:00:00:00:01:01
[01:45:07] 12_compute_probe | runs [('base', 'A'), ('base', 'B'), ('base', 'C'), ('tuned', 'A'), ('tuned', 'B')] | sets ['numq_computed', 'numq_band', 'dev_score_compute', 'dev_score_judge', 'shift_full'] | C on up to 60 cases per set | torch 2.11.0+cu128 | GPUs visible 2 (using up to 2)


In [3]:
# Cell 2 — The sets: one item per question, as notebook 11 builds them; arm B's computed values from the code-built inputs

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def render(q):
    """As notebook 11: noul as true/false with descriptions, choice by labels, score by criteria strings, lowest first."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    return opts, keys


def item(c, qid, group):
    q = c["questions"][qid]
    opts, keys = render(q)
    gold = c["gold"][qid]["probabilities"]
    return {"task": "d_" + q["type"], "id": c["id"], "qid": qid, "group": group, "state": state_text(c["state"]),
            "question": q["instructions"], "options": opts, "soft": {n: gold[k] for n, k in zip(opts, keys)}}


DATE_FORMATS = ("%Y-%m-%d", "%Y-%m-%d %H:%M", "%Y-%m-%dT%H:%M", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%d %b %Y", "%d %B %Y",
                "%b %d %Y", "%B %d %Y", "%b %d, %Y", "%B %d, %Y")


def parse_date(s):
    for f in DATE_FORMATS:
        try:
            return datetime.datetime.strptime(s.strip().replace(".", ""), f)
        except ValueError:
            pass
    return None


def computed_value(info):
    """The value a code-built question turns on, as one plain line (arm B), or None."""
    t, p, v = info["template"], info["paths"], info["values"]
    name = lambda path: path.split(".")[-1].split("]")[-1].replace("_", " ")
    fmt = lambda x: f"{x:.1f}".rstrip("0").rstrip(".")
    if t == "dates":
        a, b = parse_date(v[0]), parse_date(v[1])
        if a is None or b is None:
            return None
        gap = b - a
        days = gap.days if not gap.seconds else fmt(gap.total_seconds() / 86400)
        return f"{name(p[1])} is {days} days after {name(p[0])}"
    if t == "ratio":
        return f"{name(p[0])} is {fmt(100 * float(v[0]) / float(v[1]))}% of {name(p[1])}"
    if t == "difference":
        return f"{name(p[0])} minus {name(p[1])} is {fmt(float(v[0]) - float(v[1]))}"
    return None


numq_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / "v1_numq_u" / "heldout_final.jsonl")]
HINT = {}
SET_ITEMS = {"numq_computed": [], "numq_band": [], "dev_score_compute": [], "dev_score_judge": []}
for c in numq_cases:
    for qid, info in c["meta"]["numq"].items():
        if info["template"] in ("dates", "ratio", "difference"):
            h = computed_value(info)
            assert h is not None, (c["id"], qid, info["values"])
            g = c["gold"][qid]["probabilities"]
            assert (info["margin"] > 1) == (max(g, key=g.get) == "true"), (c["id"], qid)   # the value agrees with the gold
            HINT[(c["id"], qid)] = h
            SET_ITEMS["numq_computed"].append(item(c, qid, info["template"]))
        elif info["template"] == "band":
            SET_ITEMS["numq_band"].append(item(c, qid, "band"))

d_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / D_SET / "systemone.jsonl")]


def dev_split(ids):
    ids = sorted(ids)
    random.Random(42).shuffle(ids)
    return set(ids[: int(len(ids) * DEV_FRACTION)])


ADDED, TRAIN_ONLY = ("I", "K", "L", "M"), ("T", "U")
dev_ids = dev_split(c["id"] for c in d_cases if not c["id"].startswith(ADDED + TRAIN_ONLY))
for g in ADDED:
    dev_ids |= dev_split(c["id"] for c in d_cases if c["id"].startswith(g))
for c in d_cases:
    if c["id"] not in dev_ids:
        continue
    for qid, q in c["questions"].items():
        kind = c["meta"]["kinds"].get(qid, "?")
        if q["type"] == "score":
            SET_ITEMS["dev_score_compute" if kind in ("compare", "infer") else "dev_score_judge"].append(item(c, qid, kind))


def cases_of(items):
    """Items -> {case id: [its items]} in order."""
    out = {}
    for it in items:
        out.setdefault(it["id"], []).append(it)
    return out


def c_sample(set_name):
    ids = sorted(cases_of(SET_ITEMS[set_name]))
    return set(ids if len(ids) <= C_MAX_CASES else random.Random(0).sample(ids, C_MAX_CASES))


shift_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / "independent_v1" / "shift_dev.jsonl")]
SET_ITEMS["shift_full"] = [item(c, qid, c["meta"]["kinds"].get(qid, "?")) for c in shift_cases for qid in c["questions"]]
C_CASES = {s: c_sample(s) for s in SET_ITEMS}
for s, its in SET_ITEMS.items():
    log(f"{s}: {len(its)} questions in {len(cases_of(its))} cases | groups {dict(Counter(it['group'] for it in its))} | "
        f"arm C on {len(C_CASES[s])} cases")
ex = SET_ITEMS["numq_computed"][0]
log(f"arm B example: {ex['question']} -> added line: {HINT[(ex['id'], ex['qid'])]}")

From https://github.com/skamalj/llms-the-hard-way
 * branch            signpost-typed-decisions -> FETCH_HEAD
Reset branch 'signpost-typed-decisions'


Your branch is up to date with 'origin/signpost-typed-decisions'.
[01:45:08] numq_computed: 96 questions in 52 cases | groups {'dates': 79, 'ratio': 9, 'difference': 8} | arm C on 52 cases
[01:45:08] numq_band: 101 questions in 101 cases | groups {'band': 101} | arm C on 60 cases
[01:45:08] dev_score_compute: 291 questions in 213 cases | groups {'infer': 115, 'compare': 176} | arm C on 60 cases
[01:45:08] dev_score_judge: 399 questions in 288 cases | groups {'judge': 343, 'read': 56} | arm C on 60 cases
[01:45:08] shift_full: 1833 questions in 408 cases | groups {'compare': 704, 'infer': 601, 'judge': 483, 'read': 45} | arm C on 60 cases
[01:45:08] arm B example: The guest's departure date is more than 2 days after arrival. -> added line: departure is 3 days after arrival


In [4]:
# Cell 3 — The model: notebook 11's packed prompt (arms A and B), the same questions with working allowed (arm C)

from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM_PACKED = ("You read a case and answer every decision question about it. For each question, write its number and the "
                 "letter of the option you choose.")
SYSTEM_REASON = ("You read a case and answer every decision question about it. First work through each question briefly, "
                 "doing any arithmetic step by step. Then end with a line 'Answers:' followed by one line per question: its "
                 "number, a colon and the letter of the option you choose, for example '1: B'.")
ANSWER_LINE = re.compile(r"^\s*\**\[?(\d+)\]?\**\s*[:.)]\s*\**\s*\(?([A-Z])\b", re.M)


def question_block(item):
    """As notebook 11 (score levels with their positions)."""
    names, task = list(item["options"]), item["task"]
    if task == "d_noul":
        head = f"Statement: {item['question']}\nIs the statement true or false?"
    elif task == "d_score":
        head = f"Question: {item['question']}\nLevels, lowest first:"
    else:
        head = f"Question: {item['question']}\nOptions:"
    n, lines = len(names), []
    for i, nm in enumerate(names):
        pos = ""
        if task == "d_score":
            pos = f"level {i + 1} of {n}" + (" (lowest)" if i == 0 else " (highest)" if i == n - 1 else "") + ": "
        desc = item["options"][nm]
        lines.append(f"{LETTERS[i]}. {pos}{nm}" + (f": {desc}" if desc else ""))
    return head + "\n" + "\n".join(lines)


def user_text(case_items, arm):
    state = case_items[0]["state"]
    if arm == "B":
        state += "\n\nWorked figures:\n" + "\n".join(f"- {HINT[(it['id'], it['qid'])]}" for it in case_items)
    return "Case:\n" + state + "\n\n" + "\n\n".join(f"[{i + 1}] " + question_block(it) for i, it in enumerate(case_items))


def parse_answers(text, n):
    """Letters per question from arm C's text: the last 'Answers:' block; questions missing there take their last 'n: X'
    anywhere in the text."""
    block = text.rsplit("Answers:", 1)[-1] if "Answers:" in text else ""
    got = {int(m.group(1)): m.group(2) for m in ANSWER_LINE.finditer(block)}
    if len(got) < n:
        got = {**{int(m.group(1)): m.group(2) for m in ANSWER_LINE.finditer(text)}, **got}
    return [got.get(i + 1) for i in range(n)]


class Decoder:
    def __init__(self):
        name = MODEL["name"]
        self.rev = api.model_info(name, token=HF_TOKEN).sha
        self.tok = AutoTokenizer.from_pretrained(name, revision=self.rev, token=HF_TOKEN)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.m = AutoModelForCausalLM.from_pretrained(name, revision=self.rev, token=HF_TOKEN, dtype=torch.float16,
                                                      device_map={"": DEVICE.index or 0} if AMP else None).eval()
        self.inner = self.m.model.language_model if hasattr(self.m.model, "language_model") else self.m.model
        self.head_w = self.m.get_output_embeddings().weight
        ids = [self.tok(" " + L, add_special_tokens=False)["input_ids"] for L in LETTERS]
        assert all(len(x) == 1 for x in ids)
        self.label_ids = torch.tensor([x[0] for x in ids], device=DEVICE)
        log(f"loaded {name} @ {self.rev[:8]} | fp16")

    def prefix(self, case_items, arm):
        msgs = [{"role": "system", "content": SYSTEM_REASON if arm == "C" else SYSTEM_PACKED},
                {"role": "user", "content": user_text(case_items, arm)}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def one_pass(self, case_items, arm):
        """Arms A and B: notebook 11's packed pass -> one {option: probability} per question."""
        ids, pos = self.tok(self.prefix(case_items, arm), add_special_tokens=False)["input_ids"], []
        for i in range(len(case_items)):
            ids = ids + self.tok(f"{i + 1}:", add_special_tokens=False)["input_ids"]
            pos.append(len(ids) - 1)
            ids = ids + self.tok(" _\n", add_special_tokens=False)["input_ids"]
        assert len(ids) <= MAX_PROMPT_TOKENS, f"a case of {len(ids)} tokens"
        x = torch.tensor([ids], device=DEVICE)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            h = self.inner(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).last_hidden_state
        logits = h[0, pos].float() @ self.head_w[self.label_ids].float().T
        out = []
        for i, it in enumerate(case_items):
            p = torch.softmax(logits[i, : len(it["options"])], 0).tolist()
            out.append(dict(zip(it["options"], p)))
        return out, {"prompt_tokens": len(ids)}

    def reason(self, case_items, arm):
        """Arm C: greedy generation, letters parsed from the 'Answers:' block -> one {option: 1.0} (or None) per question."""
        ids = self.tok(self.prefix(case_items, "C"), add_special_tokens=False)["input_ids"]
        x = torch.tensor([ids], device=DEVICE)
        with torch.no_grad():
            y = self.m.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                temperature=None, top_p=None, top_k=None, use_cache=True, pad_token_id=self.tok.pad_token_id)
        new = y[0, len(ids):]
        text = self.tok.decode(new, skip_special_tokens=True)
        out = []
        for it, L in zip(case_items, parse_answers(text, len(case_items))):
            k = LETTERS.index(L) if L else -1
            out.append({n: float(j == k) for j, n in enumerate(it["options"])} if 0 <= k < len(it["options"]) else None)
        return out, {"prompt_tokens": len(ids), "new_tokens": len(new), "text": text,
                     "truncated": len(new) >= MAX_NEW_TOKENS}

    def merge_adapter(self, d):
        """The LoRA weight change (B @ A x alpha / r) added to the base weights in place."""
        from safetensors.torch import load_file
        conf = json.load(open(d / "adapter_config.json"))
        scale, w, n = conf["lora_alpha"] / conf["r"], load_file(str(d / "adapter_model.safetensors")), 0
        with torch.no_grad():
            for k, a in w.items():
                if ".lora_A." not in k:
                    continue
                path = k.replace("base_model.model.", "", 1).split(".lora_A.")[0]
                weight = self.m.get_submodule(path).weight
                b = w[k.replace(".lora_A.", ".lora_B.")].to(weight.device, torch.float32)
                weight += ((b @ a.to(weight.device, torch.float32)) * scale).to(weight.dtype)
                n += 1
        log(f"merged {n} adapted modules from {d}")

In [5]:
# Cell 4 — Every (model, arm, set): cases shared out over the GPUs, each timed at batch 1; results saved per run (a rerun
# skips finished ones), then the table

def spread(jobs, fn):
    mine = list(range(RANK, len(jobs), WORLD))
    res = fn([jobs[i] for i in mine])
    if WORLD == 1:
        return res
    parts = [None] * WORLD
    dist.all_gather_object(parts, list(zip(mine, res)))
    out = [None] * len(jobs)
    for part in parts:
        for i, r in part:
            out[i] = r
    return out


def find_adapter():
    d = Path("checkpoints") / TUNED_ADAPTER
    return d if (d / "adapter_model.safetensors").exists() else None


def run_one(model, kind, arm, set_name):
    f = OUT / f"{kind}_{arm}_{set_name}.json"
    if f.exists():
        log(f"{kind} {arm} {set_name}: finished earlier, read back")
        return
    by = cases_of(SET_ITEMS[set_name])
    jobs = [(cid, its) for cid, its in by.items() if arm != "C" or cid in C_CASES[set_name]]
    t_start = time.perf_counter()

    def work(js):
        out = []
        for j, (cid, its) in enumerate(js):
            if AMP:
                torch.cuda.synchronize(DEVICE)
            t0 = time.perf_counter()
            preds, extra = (model.reason if arm == "C" else model.one_pass)(its, arm)
            if AMP:
                torch.cuda.synchronize(DEVICE)
            ms = (time.perf_counter() - t0) * 1000
            rows = []
            for it, p in zip(its, preds):
                g = it["soft"]
                gold_top = max(g, key=g.get)
                top = max(p, key=p.get) if p else None
                names = list(it["options"])
                rows.append({"qid": it["qid"], "group": it["group"], "type": it["task"][2:], "ok": top == gold_top,
                             "parsed": p is not None, "pred": top, "gold": gold_top, "gold_p": round(g[gold_top], 4),
                             "gold_p_of_pred": round(g[top], 4) if top else None,
                             "level_error": abs(names.index(top) - names.index(gold_top)) if top and it["task"] == "d_score" else None,
                             "p": [round(p[n], 4) for n in names] if p else None})
            out.append({"id": cid, "ms": round(ms, 1), "questions": len(its), **extra, "rows": rows})
            if arm == "C" and RANK == 0 and (j + 1) % 10 == 0:
                log(f"  {kind} C {set_name}: {j + 1}/{len(js)} cases on GPU 0 | last {ms / 1000:.1f} s, {extra['new_tokens']} tokens")
        return out

    res = spread(jobs, work)
    if RANK == 0:
        json.dump(res, open(f, "w"), indent=1)
        acc = statistics.mean(r["ok"] for c in res for r in c["rows"])
        log(f"{kind} {arm} {set_name}: accuracy {acc:.3f} over {sum(len(c['rows']) for c in res)} questions, {len(res)} cases | "
            f"{(time.perf_counter() - t_start) / 60:.1f} min")
    barrier()


def barrier():
    if WORLD > 1:
        dist.barrier()


def run_all():
    model, merged = None, False
    for kind, arm in RUNS:
        sets = [s for s in SETS if (arm != "B" or s == "numq_computed") and (arm != "C" or s in C_SETS)]
        if all((OUT / f"{kind}_{arm}_{s}.json").exists() for s in sets):
            continue
        if kind == "tuned" and not merged:
            d = find_adapter()
            if d is None:
                log(f"tuned adapter '{TUNED_ADAPTER}' not found under checkpoints/: tuned runs skipped")
                break
            model = model or Decoder()
            model.merge_adapter(d)
            merged = True
        assert not (kind == "base" and merged), "base runs must come before tuned runs"
        model = model or Decoder()
        for s in sets:
            run_one(model, kind, arm, s)


def worker():
    global RANK, WORLD, DEVICE
    WORLD, RANK = int(os.environ.get("WORLD_SIZE", "1")), int(os.environ.get("RANK", "0"))
    if WORLD > 1:
        DEVICE = torch.device(f"cuda:{int(os.environ['LOCAL_RANK'])}")
        torch.cuda.set_device(DEVICE)
        dist.init_process_group("nccl", rank=RANK, world_size=WORLD, timeout=datetime.timedelta(hours=2), device_id=DEVICE)
    try:
        run_all()
    finally:
        if WORLD > 1:
            dist.destroy_process_group()


N_GPU = min(GPUS, torch.cuda.device_count()) if AMP else 1
log(f"starting on {N_GPU} GPU(s)")
if N_GPU > 1:
    assert not torch.cuda.is_initialized(), "CUDA was started in this kernel before the launch: restart the kernel and run all"
    from accelerate import notebook_launcher
    notebook_launcher(worker, num_processes=N_GPU, use_port="29511")
else:
    worker()

[01:45:16] starting on 2 GPU(s)
Launching training on 2 CUDAs.


tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[01:46:15] loaded Qwen/Qwen3-4B-Instruct-2507 @ cdbee75f | fp16
[01:46:15] base A numq_computed: finished earlier, read back
[01:46:15] base A numq_band: finished earlier, read back
[01:46:15] base A dev_score_compute: finished earlier, read back
[01:46:15] base A dev_score_judge: finished earlier, read back
[01:49:41] base A shift_full: accuracy 0.422 over 1833 questions, 408 cases | 3.4 min
[01:49:42] merged 252 adapted modules from checkpoints/dec-Qwen3-4B-Instruct-2507-v1_6-train-pos1/lr0.0001-r16-ep2-fp16-packed-seed42/adapter
[01:49:42] tuned A numq_computed: finished earlier, read back
[01:49:42] tuned A numq_band: finished earlier, read back
[01:49:42] tuned A dev_score_compute: finished earlier, read back
[01:49:42] tuned A dev_score_judge: finished earlier, read back
[01:53:12] tuned A shift_full: accuracy 0.727 over 1833 questions, 408 cases | 3.5 min


In [6]:
# Cell 5 — The table: accuracy (all, and on arm C's cases), by group, parse failures, latency per case, generated tokens

RANK, WORLD = 0, 1


def pct(xs, q):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(q * len(xs)))] if xs else None


def acc(rows):
    return round(statistics.mean(r["ok"] for r in rows), 3) if rows else None


SUMMARY = []
for s in SETS:
    log(f"===== {s} =====")
    for kind, arm in RUNS:
        f = OUT / f"{kind}_{arm}_{s}.json"
        if not f.exists():
            continue
        res = json.load(open(f))
        rows = [r for c in res for r in c["rows"]]
        on_c = [r for c in res if c["id"] in C_CASES[s] for r in c["rows"]]
        groups = {g: acc([r for r in rows if r["group"] == g]) for g in sorted({r["group"] for r in rows})}
        lerr = [r["level_error"] for r in rows if r["level_error"] is not None]
        line = {"set": s, "model": kind, "arm": arm, "questions": len(rows), "cases": len(res), "accuracy": acc(rows),
                "accuracy_on_C_cases": acc(on_c), "by_group": groups,
                "mean_level_error": round(statistics.mean(lerr), 3) if lerr else None,
                "unparsed": sum(not r["parsed"] for r in rows),
                "ms_per_case_p50": pct([c["ms"] for c in res], 0.5), "ms_per_case_p95": pct([c["ms"] for c in res], 0.95),
                "ms_per_question": round(sum(c["ms"] for c in res) / len(rows), 1),
                "ms_q_p50": pct([c["ms"] / c["questions"] for c in res], 0.5), "ms_q_p95": pct([c["ms"] / c["questions"] for c in res], 0.95),
                "new_tokens_mean": round(statistics.mean(c["new_tokens"] for c in res), 1) if arm == "C" else None,
                "truncated": sum(c.get("truncated", False) for c in res) if arm == "C" else None}
        SUMMARY.append(line)
        log(f"{kind:5s} {arm} | acc {line['accuracy']} (n {line['questions']}) | on C's cases {line['accuracy_on_C_cases']} | "
            f"by group {groups} | level err {line['mean_level_error']} | unparsed {line['unparsed']} | per case p50 "
            f"{line['ms_per_case_p50']:.0f} ms p95 {line['ms_per_case_p95']:.0f} ms | per question {line['ms_per_question']:.0f} ms"
            + (f" | {line['new_tokens_mean']} tokens, {line['truncated']} truncated" if arm == "C" else ""))


def bucket(gp):
    return "clear (>=0.85)" if gp >= 0.85 else "leaning (0.6-0.85)" if gp >= 0.6 else "split (<0.6)"


BREAKDOWN = {}
for kind in ("base", "tuned"):
    f = OUT / f"{kind}_A_shift_full.json"
    if not f.exists():
        continue
    res = json.load(open(f))
    rows = [r for c in res for r in c["rows"]]
    miss = [r for r in rows if not r["ok"]]
    b = {"questions": len(rows), "accuracy": acc(rows), "misses": len(miss)}
    for name in ("clear (>=0.85)", "leaning (0.6-0.85)", "split (<0.6)"):
        rs = [r for r in rows if bucket(r["gold_p"]) == name]
        ms = [r for r in miss if bucket(r["gold_p"]) == name]
        b[name] = {"questions": len(rs), "accuracy": acc(rs), "misses": len(ms),
                   "share_of_misses": round(len(ms) / max(1, len(miss)), 3),
                   "misses_where_gold_gives_our_pick_>=0.25": sum(r["gold_p_of_pred"] >= 0.25 for r in ms)}
    b["by kind"] = {k: {nm: acc([r for r in rows if r["group"] == k and bucket(r["gold_p"]) == nm])
                        for nm in ("clear (>=0.85)", "leaning (0.6-0.85)", "split (<0.6)")} | {"misses": sum(r["group"] == k for r in miss)}
                    for k in sorted({r["group"] for r in rows})}
    qms = [c["ms"] / c["questions"] for c in res]
    b["timing"] = {"questions_per_case": round(len(rows) / len(res), 2), "ms_case_p50": pct([c["ms"] for c in res], 0.5),
                   "ms_case_p95": pct([c["ms"] for c in res], 0.95), "ms_question_p50": pct(qms, 0.5), "ms_question_p95": pct(qms, 0.95),
                   "cases_over_budget": sum(q > BUDGET_MS for q in qms), "cases": len(res)}
    BREAKDOWN[kind] = b
    log(f"===== item 1a, {kind} A on all of shift_dev: accuracy {b['accuracy']} over {b['questions']} questions, {b['misses']} misses")
    for name in ("clear (>=0.85)", "leaning (0.6-0.85)", "split (<0.6)"):
        x = b[name]
        log(f"  gold {name:20s}: {x['questions']:5d} questions, accuracy {x['accuracy']}, {x['misses']} misses "
            f"({x['share_of_misses']:.0%} of all misses; gold gives our pick >= 0.25 in {x['misses_where_gold_gives_our_pick_>=0.25']})")
    log("  by kind (accuracy clear / leaning / split, misses): " + " | ".join(
        f"{k} {v['clear (>=0.85)']} / {v['leaning (0.6-0.85)']} / {v['split (<0.6)']} ({v['misses']})" for k, v in b["by kind"].items()))
    t = b["timing"]
    log(f"  timing: {t['questions_per_case']} questions per case | per case p50 {t['ms_case_p50']:.0f} ms p95 {t['ms_case_p95']:.0f} ms | "
        f"per question p50 {t['ms_question_p50']:.0f} ms p95 {t['ms_question_p95']:.0f} ms (budget {BUDGET_MS}) | "
        f"{t['cases_over_budget']} of {t['cases']} cases over budget")
print("BREAKDOWN_JSON " + json.dumps(BREAKDOWN, separators=(",", ":")))
json.dump(SUMMARY, open(f"{RUN_NAME}_summary.json", "w"), indent=1)
print("PROBE_JSON " + json.dumps(SUMMARY, separators=(",", ":")))
f = OUT / "base_C_dev_score_compute.json"
if f.exists():
    print("\nArm C, the first three working-outs (dev_score_compute):")
    for c in json.load(open(f))[:3]:
        print(f"--- {c['id']} | {[(r['pred'], r['gold']) for r in c['rows']]}\n{c['text'][:1500]}\n")

[01:53:13] ===== numq_computed =====
[01:53:13] base  A | acc 0.521 (n 96) | on C's cases 0.521 | by group {'dates': 0.544, 'difference': 0.375, 'ratio': 0.444} | level err None | unparsed 0 | per case p50 296 ms p95 525 ms | per question 189 ms
[01:53:13] base  B | acc 0.698 (n 96) | on C's cases 0.698 | by group {'dates': 0.722, 'difference': 0.375, 'ratio': 0.778} | level err None | unparsed 0 | per case p50 397 ms p95 550 ms | per question 218 ms
[01:53:13] base  C | acc 0.969 (n 96) | on C's cases 0.969 | by group {'dates': 0.962, 'difference': 1, 'ratio': 1} | level err None | unparsed 2 | per case p50 33516 ms p95 93835 ms | per question 21603 ms | 599.0 tokens, 1 truncated
[01:53:13] tuned A | acc 0.625 (n 96) | on C's cases 0.625 | by group {'dates': 0.646, 'difference': 0.375, 'ratio': 0.667} | level err None | unparsed 0 | per case p50 291 ms p95 507 ms | per question 184 ms
[01:53:13] tuned B | acc 0.729 (n 96) | on C's cases 0.729 | by group {'dates': 0.759, 'difference': 